# 02 - Computer Vision

**AI-900 Domain:** Describe features of computer vision workloads on Azure

## What You'll Learn
- **Image Analysis** - Identify objects, generate tags and captions
- **Optical Character Recognition (OCR)** - Extract text from images
- How Azure AI Vision works as a pre-built AI service

> **Just click `Run All` - no coding needed!**

## Setup: Load Credentials

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(os.path.join("..", ".env"))

AI_ENDPOINT = os.environ.get("AZURE_AI_ENDPOINT", "").rstrip("/")
AI_KEY = os.environ.get("AZURE_AI_KEY", "")

if not AI_KEY:
    raise ValueError("AZURE_AI_KEY is not set! Please check your .env file.")

print("Credentials loaded successfully!")

---
## Part 1: Image Analysis

Azure AI Vision can analyze an image and tell you:
- **Caption**: A natural language description of the image
- **Tags**: Keywords that describe the image content
- **Objects**: Specific objects detected with bounding box locations

Let's analyze a sample image!

In [ ]:
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from PIL import Image
from IPython.display import display

# Create the Image Analysis client
vision_client = ImageAnalysisClient(
    endpoint=AI_ENDPOINT,
    credential=AzureKeyCredential(AI_KEY)
)

print("Vision client created!")

### Analyze a City Scene

We'll use a publicly available sample image from Microsoft to demonstrate image analysis.

In [ ]:
# Use a public sample image URL (Microsoft sample)
image_url = "https://learn.microsoft.com/en-us/azure/ai-services/computer-vision/media/quickstarts/presentation.png"

# Download and display the image
import requests
from io import BytesIO

img_data = requests.get(image_url, timeout=10).content
img = Image.open(BytesIO(img_data))
img.thumbnail((500, 500))  # Resize for display
display(img)
print("Image loaded! Analyzing...")

In [ ]:
# Analyze the image for captions, tags, and objects
result = vision_client.analyze_from_url(
    image_url=image_url,
    visual_features=[
        VisualFeatures.CAPTION,
        VisualFeatures.TAGS,
        VisualFeatures.OBJECTS,
        VisualFeatures.DENSE_CAPTIONS
    ]
)

# Display the caption
print("=" * 50)
print("IMAGE ANALYSIS RESULTS")
print("=" * 50)

if result.caption:
    print(f"\nCaption: \"{result.caption.text}\"")
    print(f"  Confidence: {result.caption.confidence:.1%}")

# Display tags
if result.tags:
    print(f"\nTags ({len(result.tags.list)} found):")
    for tag in result.tags.list[:10]:  # Show top 10
        print(f"  - {tag.name} (confidence: {tag.confidence:.1%})")

# Display detected objects
if result.objects:
    print(f"\nObjects ({len(result.objects.list)} found):")
    for obj in result.objects.list:
        box = obj.bounding_box
        print(f"  - {obj.tags[0].name} at position ({box.x}, {box.y}), size {box.width}x{box.height}")

### Dense Captions

Dense captions describe **different regions** of the image, not just the overall scene.

In [ ]:
if result.dense_captions:
    print("Dense Captions (descriptions of different image regions):")
    print("-" * 50)
    for caption in result.dense_captions.list[:5]:  # Show top 5
        print(f"  \"{caption.text}\" (confidence: {caption.confidence:.1%})")

### Analyze a Second Image - Nature Scene

Let's try a completely different type of image to see how the AI adapts.

In [ ]:
nature_url = "https://learn.microsoft.com/en-us/azure/ai-services/computer-vision/media/quickstarts/landmark.png"

# Display the image
img_data2 = requests.get(nature_url, timeout=10).content
img2 = Image.open(BytesIO(img_data2))
img2.thumbnail((500, 500))
display(img2)

# Analyze
result2 = vision_client.analyze_from_url(
    image_url=nature_url,
    visual_features=[VisualFeatures.CAPTION, VisualFeatures.TAGS]
)

print(f"\nCaption: \"{result2.caption.text}\"")
print(f"  Confidence: {result2.caption.confidence:.1%}")
print(f"\nTop Tags:")
for tag in result2.tags.list[:8]:
    print(f"  - {tag.name} ({tag.confidence:.1%})")

---
## Part 2: Optical Character Recognition (OCR)

OCR extracts **printed or handwritten text** from images. This is incredibly useful for:
- Reading signs, documents, or receipts in photos
- Digitizing handwritten notes
- Making image content searchable

Let's extract text from an image!

In [ ]:
# Use a sample image with text
text_image_url = "https://learn.microsoft.com/en-us/azure/ai-services/computer-vision/media/quickstarts/presentation.png"

# Analyze for text (READ feature)
ocr_result = vision_client.analyze_from_url(
    image_url=text_image_url,
    visual_features=[VisualFeatures.READ]
)

print("=" * 50)
print("OCR RESULTS - Text Found in Image")
print("=" * 50)

if ocr_result.read and ocr_result.read.blocks:
    for block in ocr_result.read.blocks:
        for line in block.lines:
            print(f"  Text: \"{line.text}\"")
else:
    print("  No text detected in the image.")

---
## Key Concepts for AI-900

| Concept | Description |
|---------|-------------|
| **Image Analysis** | Generates captions, tags, and detects objects in images |
| **OCR (Read)** | Extracts printed and handwritten text from images |
| **Object Detection** | Identifies objects and their locations (bounding boxes) |
| **Dense Captions** | Describes multiple regions within a single image |
| **Pre-built Service** | No training required - you send an image, get results |
| **Confidence Score** | A value from 0 to 1 indicating how sure the AI is |

### AI-900 Exam Tips
- Azure AI Vision is part of **Azure AI Services** (previously called Cognitive Services)
- OCR can handle both **printed and handwritten** text
- Image Analysis returns **confidence scores** for each prediction
- These are **pre-built models** - you don't need your own training data
- **Custom Vision** (a different service) lets you train custom image classifiers

---
**Next:** Open `03_natural_language.ipynb` to explore NLP!